In [1]:
import operator
import pickle
from functools import partial
from pathlib import Path

import jax
import jax.numpy as jnp
import jax.scipy as jsp
import matplotlib.pyplot as plt
import numpy as np
import scipy
from jax import config
from matplotlib.lines import Line2D

config.update("jax_enable_x64", True)

import os
import statistics

import polars as pl
from matplotlib.colors import CenteredNorm

from cmfa.functions import ordered_ilr_func


In [2]:
RESULTS_DIR = Path("..") / "data" / "results" / "tca_cycle"
PLOT_DIR = Path("..") / "figures"

In [3]:
true_flux = np.load(RESULTS_DIR / "true_flux.npy", allow_pickle=True).item()
true_id = np.load(RESULTS_DIR / "true_composition.npy", allow_pickle=True).item()
print(f"True flux: {true_flux}")
print(f"True isotopologue distribution: {true_id}")

True flux: {'v8': Array(0.215, dtype=float64), 'v1': Array(1., dtype=float64), 'v19': Array(0.05, dtype=float64), 'v20': Array(0.03, dtype=float64), 'v16': Array(0.07, dtype=float64), 'v17': Array(0.23, dtype=float64), 'v18': Array(0.12, dtype=float64), 'v2': Array(0.201, dtype=float64), 'v9': Array(0.296, dtype=float64), 'v3': Array(0.621, dtype=float64), 'v4': Array(0.621, dtype=float64), 'v5': Array(0.621, dtype=float64), 'v6': Array(0.592, dtype=float64), 'v11': Array(1.213, dtype=float64), 'v12': Array(0.296, dtype=float64), 'v13': Array(1.509, dtype=float64), 'v14': Array(1.735, dtype=float64), 'v15': Array(5.561, dtype=float64), 'v10': Array(0.475, dtype=float64), 'v7': Array(0.807, dtype=float64)}
True isotopologue distribution: {'lysx_123456': Array([0.01567147, 0.11709665, 0.24642148, 0.28673379, 0.21348085,
       0.10003428, 0.02056148], dtype=float64), 'valx_12345': Array([0.0128301 , 0.10251097, 0.27570439, 0.34531928, 0.21147698,
       0.05215829], dtype=float64), 'aspx

# Reference 

In [4]:
with open(RESULTS_DIR / "estimated_fluxes_reference.pkl", "rb") as f:
    estimated_fluxes_ref = pickle.load(f)

with open(RESULTS_DIR / "estimated_fluxes_full.pkl", "rb") as f:
    estimated_fluxes_full = pickle.load(f)

In [5]:
exclude = {"v1", "v16", "v17", "v18", "v19", "v20"}
flux_keys = [k for k in sorted(true_flux.keys(), key=lambda k: int(k[1:])) if k not in exclude]

rows = []
for model in ("trad", "ilr"):
    arr = np.array(estimated_fluxes_full[model])  # shape (N, n_flux)
    for i, flux_name in enumerate(flux_keys):
        col = arr[:, list(true_flux.keys()).index(flux_name)]
        rows.append({
            "model": model,
            "flux": flux_name,
            "true_value": float(true_flux[flux_name]),
            "median": float(np.median(col)),
            "min": float(np.min(col)),
            "max": float(np.max(col)),
            "q25": float(np.quantile(col, 0.25)),
            "q75": float(np.quantile(col, 0.75)),
        })

summary_df = pl.DataFrame(rows)
summary_df.write_csv(RESULTS_DIR / "estimated_fluxes_full_summary.csv")
print(summary_df.to_pandas().to_string(index=False))

model flux  true_value   median          min      max      q25      q75
 trad   v2       0.201 0.201469 1.578093e-01 0.241478 0.193226 0.209488
 trad   v3       0.621 0.621469 5.778093e-01 0.661478 0.613226 0.629488
 trad   v4       0.621 0.621469 5.778093e-01 0.661478 0.613226 0.629488
 trad   v5       0.621 0.621469 5.778093e-01 0.661478 0.613226 0.629488
 trad   v6       0.592 0.592938 5.056186e-01 0.672957 0.576452 0.608975
 trad   v7       0.807 0.809973 5.195217e-01 2.561482 0.718245 0.930912
 trad   v8       0.215 0.216067 9.136650e-17 1.981758 0.131384 0.331630
 trad   v9       0.296 0.295791 7.508211e-02 0.753144 0.248062 0.349148
 trad  v10       0.475 0.474298 2.822967e-01 0.923618 0.429835 0.524775
 trad  v11       1.213 1.214406 1.083428e+00 1.334435 1.189678 1.238463
 trad  v12       0.296 0.296469 2.528093e-01 0.336478 0.288226 0.304488
 trad  v13       1.509 1.510875 1.336237e+00 1.670913 1.477904 1.542950
 trad  v14       1.735 1.737344 1.519046e+00 1.937391 1.696130 1

In [6]:
col_rename = {
    "flux": "Flux",
    "true_value": "True value",
    "median": "Median",
    "min": "Min",
    "max": "Max",
    "q25": "Q25",
    "q75": "Q75",
}

for model, label in (("trad", "Traditional I-MFA"), ("ilr", "Compositional I-MFA")):
    df = (
        summary_df.filter(pl.col("model") == model)
        .drop("model")
        .rename(col_rename)
        .to_pandas()
    )
    latex = df.to_latex(
        index=False,
        float_format="%.3f",
        caption=f"Estimated fluxes — {label}",
        label=f"tab:fluxes_{model}",
    )
    print(f"% --- {label} ---")
    print(latex)

% --- Traditional I-MFA ---
\begin{table}
\caption{Estimated fluxes — Traditional I-MFA}
\label{tab:fluxes_trad}
\begin{tabular}{lrrrrrr}
\toprule
Flux & True value & Median & Min & Max & Q25 & Q75 \\
\midrule
v2 & 0.201 & 0.201 & 0.158 & 0.241 & 0.193 & 0.209 \\
v3 & 0.621 & 0.621 & 0.578 & 0.661 & 0.613 & 0.629 \\
v4 & 0.621 & 0.621 & 0.578 & 0.661 & 0.613 & 0.629 \\
v5 & 0.621 & 0.621 & 0.578 & 0.661 & 0.613 & 0.629 \\
v6 & 0.592 & 0.593 & 0.506 & 0.673 & 0.576 & 0.609 \\
v7 & 0.807 & 0.810 & 0.520 & 2.561 & 0.718 & 0.931 \\
v8 & 0.215 & 0.216 & 0.000 & 1.982 & 0.131 & 0.332 \\
v9 & 0.296 & 0.296 & 0.075 & 0.753 & 0.248 & 0.349 \\
v10 & 0.475 & 0.474 & 0.282 & 0.924 & 0.430 & 0.525 \\
v11 & 1.213 & 1.214 & 1.083 & 1.334 & 1.190 & 1.238 \\
v12 & 0.296 & 0.296 & 0.253 & 0.336 & 0.288 & 0.304 \\
v13 & 1.509 & 1.511 & 1.336 & 1.671 & 1.478 & 1.543 \\
v14 & 1.735 & 1.737 & 1.519 & 1.937 & 1.696 & 1.777 \\
v15 & 5.561 & 5.558 & 4.949 & 6.262 & 5.438 & 5.684 \\
\bottomrule
\end{tabular}
\e